In [4]:
import glob
import pandas as pd
import polars as pl

# 1. 12개 CSV 파일 불러오기 및 합치기
# (파일명 패턴에 맞게 경로 수정 필요, 한글 깨짐 발생 시 encoding='utf-8'로 변경)
file_paths = sorted(glob.glob('../data/25년도 공공자전거 대여이력정보/*25*.csv'))
target_columns = [
    "자전거번호", "대여일시", "대여 대여소번호", "대여 대여소명", "대여거치대", 
    "반납일시", "반납대여소번호", "반납대여소명", "반납거치대", 
    "이용시간(분)", "이용거리(M)", "생년", "성별", "이용자종류","대여대여소ID","반납대여소ID"
]

df_list = []
for file in file_paths:
    df = pl.read_csv(file, encoding="cp949", columns=target_columns, infer_schema_length=0) 
    df_list.append(df)

df_bike_history = pl.concat(df_list)
df_bike_history.write_parquet('combined_2025_bike_history_data.parquet')

In [2]:
import json
import pandas as pd

print("⏳ Parquet 대용량 데이터 로딩 및 통계 연산 시작...")

# 1. Parquet 데이터 로드
df = pd.read_parquet("../data/combined_2025_bike_history_data.parquet")

# 2. 평균 이용시간(분) 계산 (문자열 -> 숫자 변환)
# errors='coerce': 숫자 변환 불가능한 값이나 결측치를 NaN으로 처리하여 에러 방지
use_time_numeric = pd.to_numeric(df["이용시간(분)"], errors="coerce")
avg_time = int(round(use_time_numeric.mean()))

# 3. 시간대별 대여 누적 비율 계산
# 대여일시 컬럼도 datetime 타입으로 안심 변환
rent_dt = pd.to_datetime(df["대여일시"], errors="coerce")
hourly_counts = (
    df.groupby(rent_dt.dt.hour).size().reindex(range(24), fill_value=0)
)

cumulative_counts = hourly_counts.cumsum()
total_rentals = cumulative_counts.iloc[-1]
hourly_ratio = (cumulative_counts / total_rentals).round(4).tolist()

# 4. 연산 결과 집계 및 저장
stats_data = {
    "avg_use_time": avg_time,
    "hourly_ratio": hourly_ratio,
}

with open("bike_stats_2025.json", "w", encoding="utf-8") as f:
    json.dump(stats_data, f, ensure_ascii=False, indent=2)

print("✅ 전처리 완료! 'bike_stats_2025.json' 생성 완료")
print(f"   - 평균 이용시간: {avg_time}분")

⏳ Parquet 대용량 데이터 로딩 및 통계 연산 시작...
✅ 전처리 완료! 'bike_stats_2025.json' 생성 완료
   - 평균 이용시간: 21분


In [2]:
import json
import pandas as pd

print("⏳ Parquet 데이터 로딩 및 시간대별 통계 집계 중...")

# 1. 데이터 로드 및 타입 변환
df = pd.read_parquet("../data/combined_2025_bike_history_data.parquet")
df["대여일시"] = pd.to_datetime(df["대여일시"], errors="coerce")
df = df.dropna(subset=["대여일시"])

df["date"] = df["대여일시"].dt.date
df["hour"] = df["대여일시"].dt.hour
df["is_weekend"] = df["대여일시"].dt.dayofweek >= 5  # 토/일: True, 월~금: False

# 2. 평균 이용시간(분) 계산 (Standard Python int로 변환)
use_time_numeric = pd.to_numeric(df["이용시간(분)"], errors="coerce")
avg_time = int(round(use_time_numeric.mean()))

# 3. 날짜별 + 시간대별 대여 건수 집계 후, 평일/주말 평균 산출
daily_hourly = (
    df.groupby(["date", "is_weekend", "hour"]).size().reset_index(name="count")
)
avg_hourly = (
    daily_hourly.groupby(["is_weekend", "hour"])["count"]
    .mean()
    .round()
)

# int()를 사용해 np.int64 타입을 파이썬 기본 int 타입으로 변환
weekday_hourly = [int(avg_hourly.get((False, h), 0)) for h in range(24)]
weekend_hourly = [int(avg_hourly.get((True, h), 0)) for h in range(24)]

# 4. 시간대별 전체 누적 비율 계산
hourly_counts = df.groupby("hour").size().reindex(range(24), fill_value=0)
hourly_ratio = (hourly_counts.cumsum() / hourly_counts.sum()).round(4).tolist()

# float 타입 보장
hourly_ratio = [float(x) for x in hourly_ratio]

# 5. JSON 파일 저장
stats_data = {
    "avg_use_time": avg_time,
    "hourly_ratio": hourly_ratio,
    "weekday_hourly": weekday_hourly,
    "weekend_hourly": weekend_hourly,
}

with open("bike_stats_today_2025.json", "w", encoding="utf-8") as f:
    json.dump(stats_data, f, ensure_ascii=False, indent=2)

print("✅ 실제 통계 기반 bike_stats_today_2025.json 추출 완료!")

⏳ Parquet 데이터 로딩 및 시간대별 통계 집계 중...
✅ 실제 통계 기반 bike_stats_today_2025.json 추출 완료!
